# Running on Modal

## What you'll learn

- Write a **command op** — `ToolSpec` + `execute_command()` instead of `execute_function()`
- Deploy it once as a persistent Modal endpoint with `artisan modal deploy`
- Route steps to the endpoint by flipping `compute_provider="modal"`
- Understand the spawn/poll job model and per-artifact fan-out
- Authenticate clients with Modal proxy-auth tokens
- Debug endpoint execution

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Compute Routing](01-compute-routing.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No (the demo tool runs on CPU containers).


:::{note}
This tutorial requires a Modal account, Modal credentials, and a deployed
endpoint. Code cells are shown for reference and are not executed in the
docs build.
:::


In [ ]:
from __future__ import annotations

from artisan.operations.examples import DataGenerator, WaitTool
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.schemas import ComputeProvider, ModalComputeConfig
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_data, inspect_pipeline, inspect_step

In [ ]:
env = tutorial_setup("modal_execution", clean=True)
DELTA_ROOT = env.delta_root

## Command operations

Modal compute runs **command ops**: operations that wrap an external tool. A
command op declares a `ToolSpec`, a `Params` model, and an `execute_command()` —
and **no** `execute_function()`; the framework performs the command:

```python
class WaitTool(OperationDefinition):
    name = "wait_tool"
    tool = ToolSpec(executable="bash", interpreter=None)
    compute_provider = ComputeProvider(modal=ModalComputeConfig())

    inputs = {"dataset": InputSpec(artifact_type="data", required=True)}

    class Params(BaseModel):
        model_config = {"extra": "forbid"}   # the endpoint's typed schema

        seconds: int = Field(default=10, ge=1, description="...")

    params: Params = Params()

    def preprocess(self, inputs: PreprocessInput) -> dict[str, Any]:
        # one materialized path per artifact — sliced per endpoint call
        return {"dataset": PerArtifact(
            [a.materialized_path for a in inputs.input_artifacts["dataset"]]
        )}

    def execute_command(self, inputs: dict[str, Any]) -> list[str]:
        # count one tick per second, then write <stem>_waited.csv
        return [*self.tool.parts(), "-c", f'... seq 1 {self.params.seconds} ...']
```

`execute_command` is the single source of the command. Under
`compute_provider="local"` the framework runs it as a local subprocess;
under `"modal"` the deployed copy runs it in the tool's container. A tool
op's products are the files its command writes to the execute dir — memory
results and post-run glue belong in `postprocess()`.

`WaitTool` counts up once per second (`wait_tool [<container task id>]
tick 3 / 10`) and emits a marker file recording the wait, the container,
and the source artifact — so you can *watch* remote execution live and
prove the fan-out from the committed artifacts. (The full definition is
`src/artisan/operations/examples/wait_tool.py`.)

Pure-Python operations (custom `execute_function()`) run on the local provider only.


## Deploy the endpoint (once per tool)

```bash
artisan modal deploy wait_tool
```

This deploys one persistent Modal app named `artisan-tool-wait_tool`: a
**worker** (the tool's image + hardware, one job per container) behind a
lightweight **HTTP endpoint** with a typed, tool-native API. You submit a
job, poll for its result, and download the outputs — the routes are
exercised end to end in "Calling the endpoint without Artisan" below, and
[Configure Execution](../../how-to-guides/configuring-execution.md) covers
endpoint configuration.

Deploy reads the op's **class-level** config: image, volumes, secrets,
scaling, and `data_policy` from `ModalComputeConfig`;
gpu/cpu/memory/timeout from `ComputeResources`. The empty data policy
keeps inline requests available and denies every remote URI. Redeploy
after changing the op's code, image, hardware, or URI policy.

The same endpoint serves Artisan pipelines and non-Artisan callers (curl,
other repositories) alike. Callers cannot submit or widen the baked policy.

## Authentication

The endpoint requires Modal **proxy-auth tokens** (created in the Modal
dashboard under _Settings → Proxy Auth Tokens_). Clients send them as
`Modal-Key` / `Modal-Secret` headers.

Create or edit the gitignored `.env` file at the repo root with these
keys, replacing the placeholders with your own tokens:

```dotenv
MODAL_PROXY_TOKEN_ID=wk-...
MODAL_PROXY_TOKEN_SECRET=ws-...
```

Artisan discovers the tokens from the process environment first, then the
nearest `.env` file walking up from the working directory — so Jupyter
kernels, cron jobs, and IDE test runners all work without shell-inherited
exports. Environment variables of the same names override the file (CI).

A different variable prefix can be configured per op via
`ModalComputeConfig.auth_secret`. A custom `endpoint_url` is
unauthenticated by default and never receives `MODAL_PROXY`; set an
explicit prefix to authenticate it. Authenticated URLs require HTTPS,
and control requests never follow redirects.


## Hardware lives on the deployment

Worker hardware is read from the op's class-level `ComputeResources` at
deploy time:

```python
class Op(OperationDefinition):
    ...
    compute_provider = ComputeProvider(modal=ModalComputeConfig(image=OP_IMAGE))
    compute_resources = ComputeResources(gpu="A100", memory_gb=32, timeout=7200)
```

Changing hardware means redeploying — a per-step `compute_resources`
override does **not** reconfigure an already-deployed endpoint.


In [ ]:
config = ComputeProvider(active="modal", modal=ModalComputeConfig())

print(f"Active provider: {config.active}")
print(f"Available:       {config.available()}")
print(f"Worker image:    {config.modal.image}")
print(f"Poll interval:   {config.modal.poll_interval}s")

## Running a step on the endpoint

`compute_provider="modal"` is the only change — the operation, params, and
output wiring stay identical. `skip_cache=True` ensures each example
executes, including when you rerun the cell:


In [ ]:
pipeline = PipelineManager.create(
    name="modal_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

gen = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 2, "seed": 42},
)

# local first — same op, no Modal required
local_wait = pipeline.run(
    operation=WaitTool,
    skip_cache=True,
    name="wait_local",
    inputs={"dataset": gen.output("datasets")},
    params={"seconds": 1},
)

# then on the deployed endpoint — one argument changed
modal_wait = pipeline.run(
    operation=WaitTool,
    skip_cache=True,
    name="wait_modal",
    inputs={"dataset": gen.output("datasets")},
    params={"seconds": 1},
    compute_provider="modal",
)

summary = pipeline.finalize()
assert summary["overall_success"], summary
assert local_wait.disposition is StepDisposition.EXECUTED
assert modal_wait.disposition is StepDisposition.EXECUTED
for completed_step in (gen, local_wait, modal_wait):
    artifacts = inspect_step(
        DELTA_ROOT,
        completed_step.step_number,
        pipeline_run_id=pipeline.config.pipeline_run_id,
    )
    assert artifacts.height == 2, (completed_step.step_name, artifacts)
print(f"Pipeline complete: success={summary['overall_success']}")
inspect_pipeline(DELTA_ROOT, pipeline_run_id=pipeline.config.pipeline_run_id)

## What happens during a modal step

Flipping to `"modal"` changes only *where* the execute phase runs, not the
lifecycle. For each artifact the framework submits the op's `Params` and
input files to the endpoint, polls at `poll_interval` until the job
finishes, then downloads the output tar into the artifact's `execute_dir` —
recreating the exact layout of a local run. `postprocess`, lineage capture,
and recording then run unchanged. See
[Execution Flow](../../concepts/execution-flow.md) for the full routing
lifecycle.

A few operational facts matter when you run one:

- A unit of N artifacts dispatches as N concurrent endpoint calls; the
  worker runs one job per container, so Modal scales by adding containers.
- Pipeline cancellation posts `/cancel`, terminating the running containers.
- The tool log arrives with the result, not streamed live — watch progress
  from the Modal dashboard (below).

**Transport limits:** inline inputs and the output tar are bounded at
100 MB per direction. Larger complete-file inputs may use `s3://` or
presigned HTTP refs only when the class-level `data_policy` allows their
prefix or exact origin. Every ref carries `content_digest` and `size_bytes`,
and the worker verifies both before the tool runs. Large static data (model
weights) belongs on Modal Volumes (`ModalComputeConfig.volumes`), not in
the request.

## Watch endpoint calls overlap

`WaitTool` is a command op built for exactly this: it counts up once per
second (`wait_tool [<container task id>] tick 3 / 10`), so you can _watch_ it run,
and it takes a `dataset` input role, so a step with eight artifacts can
fan out as eight concurrent endpoint calls. Each container runs one job
at a time (`max_inputs=1`); Modal's autoscaler decides how many
containers to add as queue pressure grows. Configure
`ModalComputeConfig.min_containers` at deployment time if you want a
pre-warmed pool.

The demo below batches eight artifacts into one unit
(`batch_strategy={"artifacts_per_unit": 8}`) so a single unit issues eight
concurrent endpoint calls. Unit size is also the failure and caching
granularity, so workloads that need per-artifact cache hits keep one
artifact per unit instead. For how per-unit and per-artifact concurrency
compose — and the knobs that bound each — see
[Configure Execution](../../how-to-guides/configuring-execution.md).

One GPU note: the container's GPU belongs in `compute_resources.gpu`.
Setting `runner_resources.gpus` reserves a *local* GPU and serializes
the worker pool — wrong knob when execute ships to Modal (the framework
warns).

Deploy it once:

```bash
artisan modal deploy wait_tool
```

Sequential, eight 10-second waits would take at least 80 seconds.
Concurrent calls can overlap those waits; startup time and available
capacity determine the observed duration.

In [ ]:
scale = PipelineManager.create(
    name="modal_scale_out",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = scale.output

scale_gen = scale.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 8, "seed": 42},
)

# One unit can issue up to eight concurrent endpoint calls.
step = scale.run(
    operation=WaitTool,
    skip_cache=True,
    name="wait",
    inputs={"dataset": output("generate", "datasets")},
    params={"seconds": 10},
    compute_provider="modal",
    batch_strategy={"artifacts_per_unit": 8},
)

scale_summary = scale.finalize()
assert scale_summary["overall_success"], scale_summary
assert step.disposition is StepDisposition.EXECUTED
for completed_step in (scale_gen, step):
    artifacts = inspect_step(
        env.delta_root,
        completed_step.step_number,
        pipeline_run_id=scale.config.pipeline_run_id,
    )
    assert artifacts.height == 8, (completed_step.step_name, artifacts)
print("All 8 remote waits produced an output artifact.")

### Watching it run

While the cell above executes (the first run also pays for container
cold starts):

- **Modal dashboard** — open the `artisan-tool-wait_tool` app: the
  container count may climb as the autoscaler fans out, and each container's log shows its ticks
  arriving one per second.
- **CLI** — in a terminal:

  ```bash
  modal app logs artisan-tool-wait_tool
  ```

  Overlapping calls show interleaved ticks from different container task
  IDs. The autoscaler controls how many containers actually run.

### Proof in the artifacts

Each container wrote its own task id into its marker file, so the
committed artifacts identify which containers handled the eight calls.
Count their host values below; the exact number depends on available
capacity and container reuse.


In [ ]:
markers = inspect_data(
    env.delta_root,
    step_number=step.step_number,
    pipeline_run_id=scale.config.pipeline_run_id,
)
assert markers.height == 8
assert markers["seconds"].to_list() == [10] * 8
assert all(markers["host"].to_list())
hosts = set(markers["host"].to_list())
print(f"distinct containers observed: {len(hosts)}")
print(sorted(hosts))

## Calling the endpoint without Artisan

The endpoint is a plain HTTP API — Artisan is one client among others.
Each multipart file part is named by its input *role* (`dataset`), and
`input_filenames` maps the role to the real file name so the worker
materializes it under its original stem:

```bash
URL=https://<workspace>--artisan-tool-wait-tool.modal.run
printf 'a,b\n1,2\n' > sample.csv

# discover the request contract: params schema + input roles
curl "$URL/schema" \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET"
# → {"operation": "wait_tool", "params_schema": {…}, "inputs": {"dataset": {…}}}

# submit
curl -X POST "$URL/submit" \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" \
  -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET" \
  -F 'params={"seconds": 3}' \
  -F 'input_filenames={"dataset": "sample.csv"}' \
  -F 'files=@sample.csv;filename=dataset'
# → {"call_id": "fc-..."}

# poll
curl "$URL/result?call_id=fc-..." \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET"

# download outputs (sample_waited.csv)
curl -o outputs.tar "$URL/download?call_id=fc-..." \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET"
```


## Debugging

| Problem                            | Cause                                           | Fix                                                                             |
| ---------------------------------- | ----------------------------------------------- | ------------------------------------------------------------------------------- |
| `tool_endpoint_misconfigured`      | Op is not a command op, or modal config missing    | Declare `ToolSpec` + `execute_command()`; configure `compute_provider.modal`      |
| "has no web URL — is it deployed?" | Endpoint not deployed                           | `artisan modal deploy <op>`                                                     |
| HTTP 407/401 at submit             | Missing or invalid proxy-auth tokens            | Set `MODAL_PROXY_TOKEN_ID` / `MODAL_PROXY_TOKEN_SECRET`                         |
| HTTP 422 at submit                 | Params don't match the op's schema              | The endpoint validates against `Params` (`extra="forbid"`)                      |
| `op_execute_failed`                | The tool exited non-zero                        | The error carries the stderr tail; full log in the parquet `tool_output` column |
| Result `expired`                   | Output polled more than 7 days after completion | Re-run the step                                                                 |

Develop locally first: run with `compute_provider="local"` until the
pipeline logic is correct, then flip to `"modal"`. Same op, same
`execute_command`, same outputs.


## Summary

| Concept                     | What it does                                                |
| --------------------------- | ----------------------------------------------------------- |
| Command op                  | `ToolSpec` + `Params` + `execute_command()`; no `execute_function()`   |
| `artisan modal deploy <op>` | One-time deploy: persistent worker + HTTP endpoint per tool |
| `compute_provider="modal"`  | Route a step's per-artifact execute-phase calls to the endpoint |
| Spawn/poll                  | `/submit` → `call_id`; poll `/result`; `/download` outputs  |
| `ComputeResources`          | Worker hardware, read at deploy time                        |
| Proxy auth                  | `MODAL_PROXY_TOKEN_ID` / `MODAL_PROXY_TOKEN_SECRET` headers |
| Inline transport            | ≤100 MB per direction; `s3://` URIs bypass the bound        |

Operations, inputs, params, and output wiring are identical whether the
tool runs locally or on Modal. Results land in the same Delta Lake tables
regardless.


## Next steps

- [Compute Routing](01-compute-routing.ipynb) — Step runners vs compute providers
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Complete configuration reference
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work
